In [1]:
import pandas as pd
# je n'ai chargé que les 100000 premières lignes car le fichier est trop volumineux et j'ai aussi sauté les lignes compliquees a traiter
music_set = pd.read_csv('spotify_dataset.csv', nrows=100000,  on_bad_lines='skip')
music_set.to_csv('spotify_subset.csv', index=False) #nouveau fichier de travail avec le subset sans nouvel index
music_sub_set = pd.read_csv('spotify_subset.csv')

In [2]:
music_sub_set.isnull().sum() 

user_id             0
 "artistname"      84
 "trackname"        0
 "playlistname"     5
dtype: int64

In [3]:
music_sub_set = music_sub_set.dropna() #j'ai regardé et supprimé les lignes avec des valeurs NaN


In [4]:
# J'ai nettoyé le nom car j'avais du mal à intéragir avec
music_sub_set.columns = [col.strip().replace('"', '') for col in music_sub_set.columns]
print("Nouveaux noms de colonnes :", music_sub_set.columns.tolist())

Nouveaux noms de colonnes : ['user_id', 'artistname', 'trackname', 'playlistname']


In [5]:
# J'ai ensuite vérifier qu'on avait bien 40 transactions différentes afin que les outputs de l'algorithme soient cohérents
n_users = music_sub_set['user_id'].nunique()
n_playlists = music_sub_set['playlistname'].nunique()

print(f"Nombre d'utilisateurs uniques : {n_users}")
print(f"Nombre de playlists uniques : {n_playlists}")

Nombre d'utilisateurs uniques : 143
Nombre de playlists uniques : 1902


In [6]:
# J'ai séparé les noms d'artistes par la virgule (dans le cas d'un featuring) et "aplatit" la liste
music_sub_set['artistname'] = music_sub_set['artistname'].str.split(', ')

# j'ai ensuite vérifier que chaque artiste du featuring devienne un item individuel
transactions = music_sub_set.explode('artistname').groupby('playlistname')['artistname'].apply(list).values.tolist()
transactions = [list(set(t)) for t in transactions] #meilleur reflet des associations d'artistes

In [7]:
from mlxtend.preprocessing import TransactionEncoder #import de la bibliothèque d'algorithme apriori

te = TransactionEncoder() 
te_ary = te.fit(transactions).transform(transactions) #encodage binaire des artistes si ils sont dans une playlist ou non
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

In [9]:
from mlxtend.frequent_patterns import apriori

# j'ai générer les ensembles d'articles fréquents qui respectent le min_sup
frequent_itemsets = apriori(df_encoded, min_support=0.005, use_colnames=True) #les artistes apparaissant dans moins de 0.5 % des playlists est éliminé
print(f"Nombre d'ensembles fréquents trouvés : {len(frequent_itemsets)}") 

Nombre d'ensembles fréquents trouvés : 1263


In [10]:
from mlxtend.frequent_patterns import association_rules

if not frequent_itemsets.empty: #phase d'ajustement des paramètres de support minimum afin de trouver des valeurs cohérentes
    my_rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1) #j'ai garde 10% comme valeur minimum de confiance 
    print(my_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
else:
    print("min_support trop élevé")

              antecedents                              consequents   support  \
0                 (JAY Z)                                   (2Pac)  0.005783   
1                  (2Pac)                                  (JAY Z)  0.005783   
2              (Coldplay)                      (A Great Big World)  0.005258   
3     (A Great Big World)                               (Coldplay)  0.005258   
4             (Passenger)                      (A Great Big World)  0.005783   
...                   ...                                      ...       ...   
1681  (JAY Z, Kanye West)              (M.I.A., Justin Timberlake)  0.005258   
1682             (M.I.A.)   (Justin Timberlake, JAY Z, Kanye West)  0.005258   
1683  (Justin Timberlake)              (M.I.A., JAY Z, Kanye West)  0.005258   
1684              (JAY Z)  (M.I.A., Justin Timberlake, Kanye West)  0.005258   
1685         (Kanye West)       (M.I.A., Justin Timberlake, JAY Z)  0.005258   

      confidence       lift  
0       0

In [21]:
top_rules = my_rules.sort_values(by='confidence', ascending=False)
print(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)) #affichage des meilleures associations trouvées

                                 antecedents  \
538                                  (Earth)   
539                            (Wind & Fire)   
174                        (Backstreet Boys)   
175                                 (N Sync)   
1542    (Fatboy Slim, Red Hot Chili Peppers)   
1543                           (The Prodigy)   
1681         (Kanye West, Justin Timberlake)   
1676                         (JAY Z, M.I.A.)   
1663               (Passenger, Kimya Dawson)   
1666  (Everything But The Girl, St. Vincent)   

                                 consequents   support  confidence        lift  
538                            (Wind & Fire)  0.005783    0.916667  158.500000  
539                                  (Earth)  0.005783    1.000000  158.500000  
174                                 (N Sync)  0.005258    0.833333  113.214286  
175                        (Backstreet Boys)  0.005258    0.714286  113.214286  
1542                           (The Prodigy)  0.005258    1.000000

In [22]:
top_rules.to_csv('resultats_apriori.csv', index=False) #sauvegarde des résultats 